# Workshop 001 — Reproduce an Economics Result from Raw Data

**Status:** participant material for the scheduled 23 August 2026 session.  
**Data:** synthetic teaching panel; no claim about a real policy.

This notebook validates the public data contract, reconstructs the four means, calculates the simple difference-in-differences estimate, and checks the equivalent saturated regression.

## 1. Locate and read the raw snapshot

The cell works when the notebook is opened from this folder or from the repository root. It does not edit the raw CSV.

In [ ]:
from pathlib import Path
import csv

candidates = [
    Path('data/panel.csv'),
    Path('programs/modern/workshops/001/data/panel.csv'),
]
DATA = next((p for p in candidates if p.exists()), None)
assert DATA is not None, 'Could not locate data/panel.csv'

with DATA.open(newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

print(f'Loaded {len(rows)} rows from {DATA}')

## 2. Enforce the data contract

Do not proceed if a check fails. First decide whether the input is wrong, the documentation is wrong, or the release has changed.

In [ ]:
required = {'region', 'year', 'employment_rate', 'treated', 'post'}
assert len(rows) == 56
assert required.issubset(rows[0])
assert all(all(str(r[k]).strip() for k in required) for r in rows)
assert len({(r['region'], r['year']) for r in rows}) == 56
assert all(int(r['post']) == (int(r['year']) >= 2022) for r in rows)

treatment_by_region = {}
for row in rows:
    treatment_by_region.setdefault(row['region'], set()).add(row['treated'])
assert all(len(values) == 1 for values in treatment_by_region.values())
print('Data contract: PASS')

## 3. Create the interaction in code

The raw file stays unchanged.

In [ ]:
for row in rows:
    row['employment_rate'] = float(row['employment_rate'])
    row['treated'] = int(row['treated'])
    row['post'] = int(row['post'])
    row['did'] = row['treated'] * row['post']

## 4. Calculate the four means and DiD

Commit to your result before opening `answer_key.md`.

In [ ]:
def mean(values):
    return sum(values) / len(values)

def group_mean(treated, post):
    values = [r['employment_rate'] for r in rows if r['treated'] == treated and r['post'] == post]
    return mean(values)

means = {
    'treated_pre': group_mean(1, 0),
    'treated_post': group_mean(1, 1),
    'control_pre': group_mean(0, 0),
    'control_post': group_mean(0, 1),
}
did = (means['treated_post'] - means['treated_pre']) - (means['control_post'] - means['control_pre'])
means, did

## 5. Check the saturated regression

The pure-Python solver avoids making a package installation the first reproduction barrier. The design matrix is `1 + treated + post + treated×post`.

In [ ]:
def solve(matrix, vector):
    a = [list(row) + [value] for row, value in zip(matrix, vector)]
    n = len(a)
    for col in range(n):
        pivot = max(range(col, n), key=lambda i: abs(a[i][col]))
        assert abs(a[pivot][col]) > 1e-12
        a[col], a[pivot] = a[pivot], a[col]
        scale = a[col][col]
        a[col] = [v / scale for v in a[col]]
        for r in range(n):
            if r == col:
                continue
            factor = a[r][col]
            a[r] = [v - factor * p for v, p in zip(a[r], a[col])]
    return [row[-1] for row in a]

X = [[1.0, r['treated'], r['post'], r['did']] for r in rows]
y = [r['employment_rate'] for r in rows]
xtx = [[sum(x[i] * x[j] for x in X) for j in range(4)] for i in range(4)]
xty = [sum(x[i] * value for x, value in zip(X, y)) for i in range(4)]
beta = solve(xtx, xty)
assert abs(beta[3] - did) < 1e-9
dict(zip(['intercept', 'treated', 'post', 'treated:post'], beta))

## 6. Claim boundary

Matching the coefficient proves that this public package is internally reproducible under the stated checks. It does **not** establish parallel trends, rule out anticipation, justify conventional inference with eight clusters, or turn synthetic data into evidence about a real policy.

## 7. Blind handoff

Give a second participant only this public folder. A pass requires them to recover **56 rows**, **four means**, and **2.6679167** without verbal instructions. Record friction in `reproduction_log.csv`; do not erase the discrepancy after repairing it.